# 2. Molecular Dynamics System Preparation

This notebook prepares the protein-ligand complexes for Molecular Dynamics (MD) simulations. The workflow includes:
- Structure preparation and format conversion
- Ligand protonation at physiological pH
- Force field parameterization using GAFF2
- Complex assembly and solvation
- System equilibration setup

**Selected chalcones for MD:** 13e, 14c, 14l (highest IC50)

## 2.1 Protein Structure Preparation

Convert the FRD protein structure from MOL2 to PDB format and prepare it for MD simulations using:
- **Antechamber**: Format conversion from MOL2 to PDB
- **pdb4amber**: Clean PDB file for AMBER compatibility (removes non-standard records)
- **sed**: Remove CONNECT records that are not needed for MD

In [ ]:
# Conversion of docking structures to PDB format for MD simulations using Antechamber and pdb4amber

antechamber -i ./docking_structures/FRD-apo_H.mol2 -fi mol2 -o ./md_initial_structures/FRD-apo_H.pdb -fo pdb

pdb4amber -i ./md_initial_structures/FRD-apo_H.pdb -o ./md_initial_structures/FRD_amber.pdb

sed -i '/CONNECT/d' ./md_initial_structures/FRD_amber.pdb

## 2.2 Ligand Protonation

Add hydrogen atoms to the docked chalcone structures at physiological pH (7.4) using Open Babel.

The script processes three chalcones: **13e**, **14c**, and **14l**

In [ ]:
%%writefile chalcones_protonation.sh

#!/bin/bash

mkdir -p ./md_initial_structures

# Protonation of docking chalcones for MD simulations using Open Babel

obabel ./docking_structures/13e.pdb -O ./md_initial_structures/13e.pdb -h
obabel ./docking_structures/14c.pdb -O ./md_initial_structures/14c.pdb -h
obabel ./docking_structures/14l.pdb -O ./md_initial_structures/14l.pdb -h

# Remove CONNECT records from the PDB files

sed -i '/CONNECT/d' ./md_initial_structures/13e.pdb
sed -i '/CONNECT/d' ./md_initial_structures/14c.pdb
sed -i '/CONNECT/d' ./md_initial_structures/14l.pdb

In [ ]:
bash chalcones_protonation.sh

## 2.3 Force Field Parameterization

Generate AMBER-compatible force field parameters for the chalcone ligands using the GAFF2 (General AMBER Force Field 2) framework. This multi-step process includes:

1. **Antechamber**: Convert PDB to MOL2 format with AM1-BCC charges
2. **parmchk2**: Generate missing force field parameters (frcmod files)
3. **tleap**: Create AMBER library files for each ligand

**Output files:**
- `.mol2`: Ligand structure with atom types and charges
- `.frcmod`: Force field modification parameters
- `.lib`: AMBER library files for complex building

In [ ]:
%%writefile chalcones_parameterization.sh

#!/bin/bash

mkdir -p ./md_initial_structures/chalcones_parameters

# Parameterization of docking chalcones for MD simulations using Antechamber

antechamber -i ./md_initial_structures/13e.pdb -fi pdb -o ./md_initial_structures/13e.mol2 -fo mol2 -c bcc -s 2 gaff2
antechamber -i ./md_initial_structures/14c.pdb -fi pdb -o ./md_initial_structures/14c.mol2 -fo mol2 -c bcc -s 2 gaff2
antechamber -i ./md_initial_structures/14l.pdb -fi pdb -o ./md_initial_structures/14l.mol2 -fo mol2 -c bcc -s 2 gaff2

# Generate force field modification files

parmchk2 -i ./md_initial_structures/13e.mol2 -f mol2 -o ./md_initial_structures/chalcones_parameters/13e.frcmod -s 2
parmchk2 -i ./md_initial_structures/14c.mol2 -f mol2 -o ./md_initial_structures/chalcones_parameters/14c.frcmod -s 2
parmchk2 -i ./md_initial_structures/14l.mol2 -f mol2 -o ./md_initial_structures/chalcones_parameters/14l.frcmod -s 2

# tleap

cat <<EOF > tleap13e.in

# Load force fields
source leaprc.protein.ff19SB
source leaprc.gaff2

# Load the molecule, parameters, and check
chalcone13e = loadmol2 ./md_initial_structures/13e.mol2
loadamberparams ./md_initial_structures/chalcones_parameters/13e.frcmod
check chalcone13e

# Save the library file
saveoff chalcone13e ./md_initial_structures/chalcones_parameters/13e.lib
quit
EOF
tleap -f tleap13e.in

cat <<EOF > tleap14c.in

# Load force fields
source leaprc.protein.ff19SB
source leaprc.gaff2

# Load the molecule, parameters, and check
chalcone14c = loadmol2 ./md_initial_structures/14c.mol2
loadamberparams ./md_initial_structures/chalcones_parameters/14c.frcmod
check chalcone14c

# Save the library file
saveoff chalcone14c ./md_initial_structures/chalcones_parameters/14c.lib
quit
EOF
tleap -f tleap14c.in

cat <<EOF > tleap14l.in

# Load force fields
source leaprc.protein.ff19SB
source leaprc.gaff2

# Load the molecule, parameters, and check
chalcone14l = loadmol2 ./md_initial_structures/14l.mol2
loadamberparams ./md_initial_structures/chalcones_parameters/14l.frcmod
check chalcone14l

# Save the library file
saveoff chalcone14l ./md_initial_structures/chalcones_parameters/14l.lib
quit
EOF
tleap -f tleap14l.in

# Cleanup intermediate files
rm tleap13e.in tleap14c.in tleap14l.in

In [ ]:
bash chalcones_parameterization.sh

## 2.4 Build Complete MD Systems

Assemble the complete solvated systems for MD simulations.

**For each chalcone (13e, 14c, 14l):**
1. **Load force fields**: ff19SB (protein), GAFF2 (ligand), OPC (water)
2. **Combine protein + ligand**: Create the complex
3. **Add solvation box**: OPC water model with 10 Å padding
4. **Neutralize system**: Add Cl⁻ counterions as needed
5. **Hydrogen mass repartitioning**: Enables 4 fs timestep with SHAKE
6. **Generate reference PDB**: Dry structure for visualization

**Output files per system:**
- `FRD-{ligand}.parm7`: Topology/parameter file
- `FRD-{ligand}.rst7`: Coordinate file
- `FRD-{ligand}_ref.pdb`: Reference structure (no solvent)



In [ ]:
%%writefile build_md_systems.sh

#!/bin/bash

mkdir -p chalcone-13e
mkdir -p chalcone-14c
mkdir -p chalcone-14l

for ligand in 13e 14c 14l; do
    cat <<EOF > build_system.in

    # Load force fields
    source leaprc.protein.ff19SB
    source leaprc.gaff2
    source leaprc.water.opc

    # Load ligand parameters
    loadoff ./md_initial_structures/chalcones_parameters/${ligand}.lib
    loadamberparams ./md_initial_structures/chalcones_parameters/${ligand}.frcmod

    # Load protein and ligand, and combine into a complex
    protein = loadpdb ./md_initial_structures/FRD_amber.pdb
    ligand = loadmol2 ./md_initial_structures/${ligand}.mol2
    complex = combine {protein ligand}

    # Add solvent box and ions
    solvatebox complex OPCBOX 10.0
    charge complex
    addIons Cl- 0
    check complex

    # save the system parameters and coordinates
    saveamberparm complex ./chalcone-${ligand}/FRD-${ligand}_no_hmass.parm7 ./chalcone-${ligand}/FRD-${ligand}.rst7
    quit
EOF
    tleap -f build_system.in

    parmed <<EOF

    # Load the system parameters and coordinates
    parm ./chalcone-${ligand}/FRD-${ligand}_no_hmass.parm7
    loadRestrt ./chalcone-${ligand}/FRD-${ligand}.rst7

    # Hidrogen mass repartitioning
    Hmassrepartition

    # Save the new parameters and coordinates
    outparm ./chalcone-${ligand}/FRD-${ligand}.parm7

    # Remove solvent and ions for visualization
    strip :WAT,Cl-

    # Save the stripped structure for visualization
    outPDB ./chalcone-${ligand}/FRD-${ligand}_ref.pdb
    quit
EOF

# Cleanup intermediate files
rm build_system.in ./chalcone-*/FRD-*_no_hmass.parm7

done

In [ ]:
bash build_md_systems.sh